In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit_probe_epoch50_patience10 import ViViTBackbone, ViViTLinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+ViViT + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:3' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 01:52:41,104]::1535783689::DEBUG::=============r2plus1d+ViViT + AST fusion after transformer blocks+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:3


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            final_data_set.extend(pickle.load(output_file))
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 50
print(1)
# Multi-seed setting
SEEDS = [42]
patience = 10

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# ViViT FCCC Layer-wise Linear Probe | Multi-seed | UDIVA / MSE
# ============================================================
backbone_checkpoint_path = './save_swintransformer_folder/model_best_multiseed.pth'
probe_best_model_path = './save_swintransformer_folder/vivit_linear_probe_best.pth'
os.makedirs(os.path.dirname(probe_best_model_path), exist_ok=True)
criterion = torch.nn.MSELoss().to(device)

def load_backbone_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    return model

seed_results = []
global_best_mean_mse = float('inf')
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(3):
    for seed in SEEDS:
        print('\n' + '=' * 80)
        print(f'Start ViViT FCCC probe | seed = {seed}')
        print('=' * 80)
        set_seed(seed)

        train_generator = torch.Generator(); train_generator.manual_seed(seed)
        val_generator = torch.Generator(); val_generator.manual_seed(seed + 10000)
        train_dataloader = DataLoader(dataset=train_set_data,batch_size=batchsz,shuffle=True,
            num_workers=num_workerssz,worker_init_fn=seed_worker,generator=train_generator)
        val_dataloader = DataLoader(dataset=val_set_data,batch_size=batchsz,shuffle=False,
            num_workers=num_workerssz,worker_init_fn=seed_worker,generator=val_generator)

        backbone = load_backbone_checkpoint(
            ViViTBackbone(2,14,14,2,2,2,5,96,6,8,384,model=3),
            backbone_checkpoint_path
        )
        model = ViViTLinearProbe(backbone=backbone,num_outputs=5).to(device)
        optimizer = torch.optim.AdamW(model.probe_heads.parameters(),lr=lr)
        probe_names = list(model.probe_heads.keys())

        seed_best = {name:{'mse':float('inf'),'epoch':0} for name in probe_names}
        seed_best_mean_mse = float('inf')
        no_improve_count = 0

        print('Backbone trainable params:',
              sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
        print('Probe dimensions:',model.probe_dims)
        print(f'Metric: MSE | Patience: {patience}')

        for epoch in range(epochs):
            # ---------------- Train probe heads only ----------------
            model.train()
            train_sq_sum = {name:0.0 for name in probe_names}
            train_count = {name:0 for name in probe_names}

            for audio, fullshot, big_five_data in tqdm(
                train_dataloader,desc=f'Probe Seed {seed} Train {epoch+1}/{epochs}'
            ):
                fullshot = fullshot.permute(0,4,1,2,3)
                big_five_data = big_five_data.permute(0,2,1).squeeze()
                if big_five_data.ndim == 1:
                    big_five_data = big_five_data.unsqueeze(0)
                audio,fullshot,big_five_data = (
                    audio.to(device),fullshot.to(device),big_five_data.to(device)
                )

                optimizer.zero_grad()
                outputs = model(fullshot,audio)
                losses = {name:criterion(outputs[name],big_five_data) for name in probe_names}
                torch.stack([losses[name] for name in probe_names]).mean().backward()
                optimizer.step()

                with torch.no_grad():
                    for name in probe_names:
                        train_sq_sum[name] += torch.square(
                            outputs[name]-big_five_data
                        ).sum().item()
                        train_count[name] += big_five_data.numel()

            train_mse = {
                name:train_sq_sum[name]/train_count[name]
                for name in probe_names
            }

            # ---------------- Validation ----------------
            model.eval()
            val_sq_sum = {name:0.0 for name in probe_names}
            val_count = {name:0 for name in probe_names}
            val_trait_sq_sum = {
                name:np.zeros(5,dtype=np.float64)
                for name in probe_names
            }
            val_sample_count = 0

            with torch.no_grad():
                for audio, fullshot, big_five_data in tqdm(
                    val_dataloader,desc=f'Probe Seed {seed} Val {epoch+1}/{epochs}'
                ):
                    fullshot = fullshot.permute(0,4,1,2,3)
                    big_five_data = big_five_data.permute(0,2,1).squeeze()
                    if big_five_data.ndim == 1:
                        big_five_data = big_five_data.unsqueeze(0)
                    audio,fullshot,big_five_data = (
                        audio.to(device),fullshot.to(device),big_five_data.to(device)
                    )

                    outputs = model(fullshot,audio)
                    val_sample_count += big_five_data.shape[0]

                    for name in probe_names:
                        sq_err = torch.square(outputs[name]-big_five_data)
                        val_sq_sum[name] += sq_err.sum().item()
                        val_count[name] += big_five_data.numel()
                        val_trait_sq_sum[name] += (
                            sq_err.sum(dim=0).detach().cpu().numpy()
                        )

            val_mse = {
                name:val_sq_sum[name]/val_count[name]
                for name in probe_names
            }
            val_trait_mse = {
                name:val_trait_sq_sum[name]/val_sample_count
                for name in probe_names
            }
            mean_val_mse = float(np.mean([val_mse[name] for name in probe_names]))

            for name in probe_names:
                if val_mse[name] < seed_best[name]['mse']:
                    seed_best[name]['mse'] = float(val_mse[name])
                    seed_best[name]['epoch'] = epoch+1

            if mean_val_mse < seed_best_mean_mse:
                seed_best_mean_mse = mean_val_mse
                no_improve_count = 0

                if mean_val_mse < global_best_mean_mse:
                    global_best_mean_mse = mean_val_mse
                    global_best_seed = seed
                    global_best_epoch = epoch+1
                    torch.save({
                        'seed':seed,
                        'epoch':epoch+1,
                        'probe_model':model.probe_heads.state_dict(),
                        'mean_val_mse':mean_val_mse,
                        'per_probe_val_mse':val_mse,
                        'probe_dims':model.probe_dims
                    },probe_best_model_path)
            else:
                no_improve_count += 1

            print('\n'+'-'*132)
            print(
                f'Probe Seed {seed} | Epoch {epoch+1:03d} | '
                f'Mean Val MSE={mean_val_mse:.6f} | '
                f'Best Mean Val MSE={seed_best_mean_mse:.6f} | '
                f'Patience={no_improve_count}/{patience}'
            )

            for name in probe_names:
                o_mse,c_mse,e_mse,a_mse,n_mse = val_trait_mse[name]
                print(
                    f'{name:<14} | Train MSE {train_mse[name]:.6f} | '
                    f'Val MSE {val_mse[name]:.6f} | '
                    f'Best MSE {seed_best[name]["mse"]:.6f} | '
                    f'O MSE {o_mse:.6f} | C MSE {c_mse:.6f} | '
                    f'E MSE {e_mse:.6f} | A MSE {a_mse:.6f} | N MSE {n_mse:.6f}'
                )

            if no_improve_count >= patience:
                print(
                    f'Early stopping | seed={seed} | epoch={epoch+1} | '
                    f'Best Mean Val MSE={seed_best_mean_mse:.6f} | '
                    f'Patience={no_improve_count}/{patience}'
                )
                break

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        seed_results.append({'seed':seed,'results':seed_best})
        print('\nBest MSE results for seed',seed)
        for name in probe_names:
            print(
                f'{name:<14} Best Epoch {seed_best[name]["epoch"]:>3} | '
                f'Best MSE {seed_best[name]["mse"]:.6f}'
            )

        del model,backbone,optimizer,train_dataloader,val_dataloader,train_generator,val_generator
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print('\n'+'='*80)
print('FINAL ViViT FCCC LINEAR PROBE RESULTS -- UDIVA / MSE')
print('='*80)
probe_names = list(seed_results[0]['results'].keys())

for name in probe_names:
    mses = np.array(
        [r['results'][name]['mse'] for r in seed_results],
        dtype=np.float64
    )
    mse_std = mses.std(ddof=1) if len(mses)>1 else 0.0
    print(f'{name:<14} MSE {mses.mean():.6f} ± {mse_std:.6f}')

print('-'*80)
print(
    f'Global best mean probe MSE: {global_best_mean_mse:.6f} | '
    f'seed={global_best_seed} | epoch={global_best_epoch}'
)
print(f'Saved probe checkpoint: {probe_best_model_path}')
print('='*80)



Start ViViT FCCC probe | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299
Backbone trainable params: 0
Probe dimensions: {'block1': 96, 'block2': 96, 'block3': 96, 'block4': 96, 'block5': 96, 'block6': 96, 'final_video': 96, 'final_cross': 96, 'final_fused': 96}
Metric: MSE | Patience: 10


Probe Seed 42 Val 1/50: 100%|██████████| 191/191 [00:21<00:00,  8.73it/s]



------------------------------------------------------------------------------------------------------------------------------------
Probe Seed 42 | Epoch 001 | Mean Val MSE=1.189375 | Best Mean Val MSE=1.189375 | Patience=0/10
block1         | Train MSE 0.876915 | Val MSE 1.148971 | Best MSE 1.148971 | O MSE 0.975514 | C MSE 1.278404 | E MSE 1.087514 | A MSE 1.166145 | N MSE 1.237279
block2         | Train MSE 1.037439 | Val MSE 1.281147 | Best MSE 1.281147 | O MSE 0.987155 | C MSE 1.377707 | E MSE 1.076348 | A MSE 1.355340 | N MSE 1.609184
block3         | Train MSE 1.183279 | Val MSE 1.198886 | Best MSE 1.198886 | O MSE 1.020921 | C MSE 1.342571 | E MSE 1.086440 | A MSE 1.508590 | N MSE 1.035910
block4         | Train MSE 1.077160 | Val MSE 1.212807 | Best MSE 1.212807 | O MSE 1.087528 | C MSE 1.228672 | E MSE 1.126633 | A MSE 1.262745 | N MSE 1.358459
block5         | Train MSE 1.009951 | Val MSE 1.163862 | Best MSE 1.163862 | O MSE 0.949646 | C MSE 1.182362 | E MSE 1.129477 | A M

Probe Seed 42 Val 2/50: 100%|██████████| 191/191 [00:21<00:00,  8.89it/s]



------------------------------------------------------------------------------------------------------------------------------------
Probe Seed 42 | Epoch 002 | Mean Val MSE=1.239532 | Best Mean Val MSE=1.189375 | Patience=1/10
block1         | Train MSE 0.292331 | Val MSE 1.257016 | Best MSE 1.148971 | O MSE 1.027594 | C MSE 1.352738 | E MSE 1.126916 | A MSE 1.362307 | N MSE 1.415526
block2         | Train MSE 0.355871 | Val MSE 1.311707 | Best MSE 1.281147 | O MSE 0.944280 | C MSE 1.374154 | E MSE 1.102885 | A MSE 1.509571 | N MSE 1.627646
block3         | Train MSE 0.417231 | Val MSE 1.228596 | Best MSE 1.198886 | O MSE 0.920045 | C MSE 1.333152 | E MSE 1.119198 | A MSE 1.645329 | N MSE 1.125254
block4         | Train MSE 0.384154 | Val MSE 1.263681 | Best MSE 1.212807 | O MSE 1.054128 | C MSE 1.281526 | E MSE 1.139666 | A MSE 1.437758 | N MSE 1.405326
block5         | Train MSE 0.381456 | Val MSE 1.240905 | Best MSE 1.163862 | O MSE 0.973401 | C MSE 1.222925 | E MSE 1.148348 | A M

...


------------------------------------------------------------------------------------------------------------------------------------
Probe Seed 42 | Epoch 009 | Mean Val MSE=1.339170 | Best Mean Val MSE=1.189375 | Patience=8/10
block1         | Train MSE 0.040615 | Val MSE 1.382590 | Best MSE 1.148971 | O MSE 1.082777 | C MSE 1.334255 | E MSE 1.273167 | A MSE 1.694077 | N MSE 1.528677
block2         | Train MSE 0.036659 | Val MSE 1.394122 | Best MSE 1.281147 | O MSE 1.085102 | C MSE 1.386960 | E MSE 1.158254 | A MSE 1.718120 | N MSE 1.622172
block3         | Train MSE 0.030527 | Val MSE 1.368385 | Best MSE 1.198886 | O MSE 1.015426 | C MSE 1.318256 | E MSE 1.238247 | A MSE 1.800655 | N MSE 1.469340
block4         | Train MSE 0.023659 | Val MSE 1.352142 | Best MSE 1.212807 | O MSE 1.086958 | C MSE 1.265901 | E MSE 1.219988 | A MSE 1.723607 | N MSE 1.464257
block5         | Train MSE 0.019364 | Val MSE 1.342274 | Best MSE 1.163862 | O MSE 1.048184 | C MSE 1.280137 | E MSE 1.182883 | A M

Probe Seed 42 Val 10/50: 100%|██████████| 191/191 [00:22<00:00,  8.58it/s]



------------------------------------------------------------------------------------------------------------------------------------
Probe Seed 42 | Epoch 010 | Mean Val MSE=1.339597 | Best Mean Val MSE=1.189375 | Patience=9/10
block1         | Train MSE 0.039542 | Val MSE 1.382766 | Best MSE 1.148971 | O MSE 1.084268 | C MSE 1.340430 | E MSE 1.262953 | A MSE 1.703268 | N MSE 1.522912
block2         | Train MSE 0.035318 | Val MSE 1.392042 | Best MSE 1.281147 | O MSE 1.087872 | C MSE 1.390223 | E MSE 1.145624 | A MSE 1.725117 | N MSE 1.611375
block3         | Train MSE 0.029438 | Val MSE 1.369117 | Best MSE 1.198886 | O MSE 1.023422 | C MSE 1.330471 | E MSE 1.226749 | A MSE 1.801785 | N MSE 1.463159
block4         | Train MSE 0.022741 | Val MSE 1.354576 | Best MSE 1.212807 | O MSE 1.089630 | C MSE 1.279363 | E MSE 1.209085 | A MSE 1.729669 | N MSE 1.465131
block5         | Train MSE 0.018629 | Val MSE 1.342816 | Best MSE 1.163862 | O MSE 1.049701 | C MSE 1.284209 | E MSE 1.185643 | A M

Probe Seed 42 Val 11/50: 100%|██████████| 191/191 [00:21<00:00,  8.72it/s]



------------------------------------------------------------------------------------------------------------------------------------
Probe Seed 42 | Epoch 011 | Mean Val MSE=1.339136 | Best Mean Val MSE=1.189375 | Patience=10/10
block1         | Train MSE 0.038548 | Val MSE 1.384419 | Best MSE 1.148971 | O MSE 1.088549 | C MSE 1.345782 | E MSE 1.263292 | A MSE 1.700591 | N MSE 1.523883
block2         | Train MSE 0.034108 | Val MSE 1.393055 | Best MSE 1.281147 | O MSE 1.091190 | C MSE 1.400745 | E MSE 1.149377 | A MSE 1.718936 | N MSE 1.605025
block3         | Train MSE 0.028459 | Val MSE 1.369614 | Best MSE 1.198886 | O MSE 1.029393 | C MSE 1.338845 | E MSE 1.226517 | A MSE 1.794010 | N MSE 1.459305
block4         | Train MSE 0.021962 | Val MSE 1.353263 | Best MSE 1.212807 | O MSE 1.093608 | C MSE 1.274831 | E MSE 1.216998 | A MSE 1.719861 | N MSE 1.461019
block5         | Train MSE 0.018016 | Val MSE 1.339339 | Best MSE 1.163862 | O MSE 1.050858 | C MSE 1.290442 | E MSE 1.178095 | A 